# Crawling Data Detik.com


Pada tahap ini dilakukan proses *web crawling* untuk memperoleh informasi berita olahraga yang tersedia pada situs Detik Sport. Data yang dikumpulkan berupa judul berita beserta informasi tautan dari halaman berita yang ditemukan.

Proses pengambilan data dilakukan dengan memanfaatkan beberapa library Python. **Requests** digunakan untuk mengakses halaman web dan mengambil isi HTML dari situs yang dituju. Selanjutnya, **BeautifulSoup** digunakan untuk membaca struktur HTML dan menemukan elemen-elemen yang diperlukan, khususnya bagian judul serta URL berita. Setelah data berhasil diperoleh, **Pandas** digunakan untuk mengorganisasi hasil crawling ke dalam bentuk DataFrame sehingga data dapat ditampilkan dan diolah dengan lebih mudah.

Dengan tahapan tersebut, informasi dari halaman Detik Sport dapat dikumpulkan secara otomatis dan disusun menjadi dataset yang lebih terstruktur untuk keperluan analisis selanjutnya.


Cell 1 — Import library

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

Cell 2 — Pengaturan crawling

In [10]:
BASE_URL = "https://sport.detik.com"
URL_INDEKS = "https://sport.detik.com/indeks"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

TARGET_DATA = 200

session = requests.Session()
session.headers.update(HEADERS)

print("Target data:", TARGET_DATA)

Target data: 200


Cell 3 — Fungsi mengambil URL berita

In [11]:
def ambil_url_berita(url_halaman):
    try:
        response = session.get(url_halaman, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        daftar_url = []

        for link in soup.find_all("a", href=True):
            href = link.get("href")

            if not href:
                continue

            # Mengambil link artikel Detik Sport
            if href.startswith("/"):
                href = BASE_URL + href

            if (
                "sport.detik.com" in href
                and href not in daftar_url
                and href.startswith("https://")
            ):
                daftar_url.append(href)

        return daftar_url

    except Exception as error:
        print("Gagal mengambil halaman:", error)
        return []

Cell 4 — Mengambil URL dari beberapa halaman indeks

In [12]:
semua_url = []

for halaman in range(1, 50):

    if halaman == 1:
        url = URL_INDEKS
    else:
        url = f"{URL_INDEKS}/{halaman}"

    print(f"Mengambil halaman indeks {halaman}...")

    url_ditemukan = ambil_url_berita(url)

    for item in url_ditemukan:
        if item not in semua_url:
            semua_url.append(item)

    print("Jumlah URL terkumpul:", len(semua_url))

    time.sleep(1)

    if len(semua_url) >= TARGET_DATA * 2:
        break

print("\nTotal URL yang terkumpul:", len(semua_url))

Mengambil halaman indeks 1...
Jumlah URL terkumpul: 44
Mengambil halaman indeks 2...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/2
Jumlah URL terkumpul: 44
Mengambil halaman indeks 3...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/3
Jumlah URL terkumpul: 44
Mengambil halaman indeks 4...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/4
Jumlah URL terkumpul: 44
Mengambil halaman indeks 5...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/5
Jumlah URL terkumpul: 44
Mengambil halaman indeks 6...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/6
Jumlah URL terkumpul: 44
Mengambil halaman indeks 7...
Gagal mengambil halaman: 404 Client Error: Not Found for url: https://sport.detik.com/indeks/7
Jumlah URL terkumpul: 44
Mengambil halaman indeks 8...
Gagal mengambil

Cell 5 — Fungsi mengambil isi berita

In [13]:
def ambil_isi_berita(url):
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Beberapa kemungkinan class untuk isi artikel
        pola_konten = [
            "detail__body-text",
            "detail__body",
            "itp_bodycontent"
        ]

        konten = None

        for pola in pola_konten:
            konten = soup.find(
                "div",
                class_=lambda value: value and pola in value
            )

            if konten:
                break

        if konten is None:
            return None

        paragraf = konten.find_all("p")

        teks = []

        for p in paragraf:
            isi_p = p.get_text(" ", strip=True)

            if isi_p:
                teks.append(isi_p)

        hasil = " ".join(teks)

        # Membersihkan spasi berlebih
        hasil = re.sub(r"\s+", " ", hasil).strip()

        return hasil if hasil else None

    except Exception as error:
        return None

Cell 6 — Crawling isi berita sampai 200 data

In [14]:
data_berita = []

for nomor, url in enumerate(semua_url, start=1):

    if len(data_berita) >= TARGET_DATA:
        break

    isi = ambil_isi_berita(url)

    if isi:
        data_berita.append({
            "id": len(data_berita) + 1,
            "isi berita": isi,
            "label": "olahraga"
        })

        print(
            f"Data {len(data_berita)}/{TARGET_DATA} berhasil diambil"
        )
    else:
        print(f"Artikel ke-{nomor} tidak memiliki isi yang ditemukan")

    time.sleep(1)

print("\n==============================")
print("Crawling selesai")
print("Total data:", len(data_berita))
print("==============================")

Artikel ke-1 tidak memiliki isi yang ditemukan
Artikel ke-2 tidak memiliki isi yang ditemukan
Artikel ke-3 tidak memiliki isi yang ditemukan
Artikel ke-4 tidak memiliki isi yang ditemukan
Artikel ke-5 tidak memiliki isi yang ditemukan
Artikel ke-6 tidak memiliki isi yang ditemukan
Artikel ke-7 tidak memiliki isi yang ditemukan
Artikel ke-8 tidak memiliki isi yang ditemukan
Artikel ke-9 tidak memiliki isi yang ditemukan
Artikel ke-10 tidak memiliki isi yang ditemukan
Artikel ke-11 tidak memiliki isi yang ditemukan
Artikel ke-12 tidak memiliki isi yang ditemukan
Artikel ke-13 tidak memiliki isi yang ditemukan
Artikel ke-14 tidak memiliki isi yang ditemukan
Artikel ke-15 tidak memiliki isi yang ditemukan
Artikel ke-16 tidak memiliki isi yang ditemukan
Artikel ke-17 tidak memiliki isi yang ditemukan
Artikel ke-18 tidak memiliki isi yang ditemukan
Data 1/200 berhasil diambil
Data 2/200 berhasil diambil
Data 3/200 berhasil diambil
Data 4/200 berhasil diambil
Data 5/200 berhasil diambil
Data 

Cell 7 — Membuat DataFrame

In [15]:
df_berita = pd.DataFrame(
    data_berita,
    columns=["id", "isi berita", "label"]
)

df_tampil = df_berita.copy()

df_tampil["isi berita"] = df_tampil["isi berita"].str[:200] + "..."

df_tampil.head(10).style.hide(axis="index")

id,isi berita,label
1,"Asian Games 2026 tak lama lagi akan dimulai. Ketua Umum Pengurus Besar Taekwondo Indonesia (PBTI) , Letjen TNI Richard Tampubolon, menyampaikan optimismennya terkait cabang olahraga yang dia pimpin. A...",olahraga
2,PT Bank Negara Indonesia (Persero) Tbk atau BNI kembali memberikan dukungan terhadap perjuangan bulutangkis Indonesia di panggung internasional. Hal ini seiring ditetapkannya 20 pebulu tangkis yang ak...,olahraga
3,"Buat kamu yang mengikuti MotoGP dan merasa hafal dengan karakter berbagai sirkuit, ada tantangan menarik yang bisa diikuti. Bold Riders bersama detikcom menghadirkan kuis tebak sirkuit MotoGP dengan h...",olahraga
4,"Pelatih Timnas voli putra Indonesia, Reidel Toiran, memasang target medali di Asian Games 2026. Dia optimistis timnya bisa memberikan kejutan besar. Timnas voli putra Indonesia saat ini diperkuat 12 p...",olahraga
5,"Ketua Umum Komite Olimpiade Indonesia (KOI) , Raja Sapta Oktohari , memastikan atlet terlayani dengan baik setibanya di Nagoya, Jepang. Ini untuk menjawab kondisi yang tengah terjadi di kota tersebut....",olahraga
6,"Demam olahraga lari terus menjalar ke kalangan selebritas. Salah satu yang kini menggemari olahraga tersebut adalah Fanny Ghassani. Setelah dua kali menjajal race, Fanny kini naik level. Sports enthus...",olahraga
7,"Kontingen Indonesia bersiap menatap Asian Games 2026 di Jepang. Presiden RI, Prabowo Subianto , menjanjikan bonus Rp 3 miliar untuk peraih medali emas. Asian Games 2026 berlangsung di Aichi-Nagoya, Je...",olahraga
8,Rizki Juniansyah siap menjalani debut di Asian Games 2026. Peraih medali emas Olimpiade Paris 2024 itu mengaku sudah pulih 100 persen setelah sempat mengalami cedera bahu bulan lalu. Rizki akan tampil...,olahraga
9,"Chef de Mission (CdM) Indonesia Todotua Pasaribu terus memonitoring kondisi di Nagoya, Jepang jelang Asian Games 2026. Itu setelah adanya banjir besar yang menimpa kota tersebut. Hal tersebut ditekank...",olahraga
10,"Banjir besar melanda Nagoya, Jepang, menjelang Asian Games 2026. Kondisi tersebut menjadi perhatian Menteri Pemuda dan Olahraga (Menpora) Erick Thohir. Tapi ia yakin pemerintah Jepang sudah menyiapkan...",olahraga


Cell 8 — Mengecek jumlah dan kolom

In [16]:
print("Jumlah baris :", len(df_berita))
print("Jumlah kolom :", len(df_berita.columns))
print("Nama kolom   :", list(df_berita.columns))

Jumlah baris : 20
Jumlah kolom : 3
Nama kolom   : ['id', 'isi berita', 'label']


Cell 9 — Simpan menjadi Excel

In [17]:
nama_file = "hasil_crawling_detiksport.xlsx"

df_berita.to_excel(
    nama_file,
    index=False,
    engine="openpyxl"
)

print("File Excel berhasil dibuat:")
print(nama_file)

File Excel berhasil dibuat:
hasil_crawling_detiksport.xlsx


Hasil crawling yang diperoleh dari halaman Detik Sport kemudian ditampilkan dalam bentuk tabel menggunakan DataFrame Pandas. Tabel tersebut berisi kumpulan judul berita olahraga beserta URL dari masing-masing berita yang berhasil ditemukan. Data ini merupakan data awal yang selanjutnya dapat digunakan untuk proses pengolahan dan pembersihan teks sebelum dilakukan analisis pada tahap berikutnya.
